In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

In [7]:
BASE_OUTPUT = Path('/run/media/marcelo_baez/HD_Disc1/.S2F/output')
alias = ['mansonia_humeralis', 'mansonia_titillans']

In [8]:
PREDICTION_COLUMNS = ['protein_id', 'term_id', 'score']


def normalize_aliases(aliases):
    """Accept a single alias string or any iterable of aliases."""
    if isinstance(aliases, str):
        aliases = [aliases]
    return [str(alias).strip() for alias in aliases if str(alias).strip()]


def empty_predictions_frame() -> pd.DataFrame:
    return pd.DataFrame(columns=[*PREDICTION_COLUMNS, 'alias']).astype({'score': 'float32'})


def prediction_path(alias: str, base_path: Path = BASE_OUTPUT) -> Path:
    return base_path / alias / 'prediction.df'


def load_predictions(alias: str, base_path: Path = BASE_OUTPUT, nrows=None) -> pd.DataFrame:
    """Load the diffusion output for a single run as a tidy DataFrame."""
    path = prediction_path(alias, base_path)
    if not path.exists():
        raise FileNotFoundError(f'Prediction file not found for {alias}: {path}')
    df = pd.read_csv(
        path,
        sep='\t',
        header=None,
        names=PREDICTION_COLUMNS,
        usecols=[0, 1, 2],
        dtype={'protein_id': str, 'term_id': str, 'score': np.float32},
        na_filter=False,
        low_memory=False,
        memory_map=True,
        nrows=nrows,
    )
    df['alias'] = alias
    return df

In [9]:
for alias in normalize_aliases(alias):
    try:
        df = load_predictions(alias)
        print(f'Loaded {len(df)} predictions for {alias}')
    except FileNotFoundError as e:
        print(e)

Loaded 2964991 predictions for mansonia_humeralis
Loaded 3003196 predictions for mansonia_titillans


In [11]:
mansonia_humeralis_df = df[df['alias'] == 'mansonia_humeralis']
mansonia_titillans_df = df[df['alias'] == 'mansonia_titillans']

proteins_humeralis = set(mansonia_humeralis_df['protein_id'])
proteins_titillans = set(mansonia_titillans_df['protein_id'])

print(f'Number of proteins in mansonia_humeralis: {len(proteins_humeralis)}')
print(f'Number of proteins in mansonia_titillans: {len(proteins_titillans)}')


Number of proteins in mansonia_humeralis: 0
Number of proteins in mansonia_titillans: 283


In [13]:
mansonia_humeralis_df = load_predictions('mansonia_humeralis')